# Claim-B on the RAMP backbone (Colab, RTX PRO 6000)

Two arms = **full RAMP loss + a representation term reusing RAMP's own ℓ∞/ℓ₁ adversarials** (minimal extra compute):
- **B1 = RAMP + pull-push** (α·scaffold+β·glue, anchor=clean stop-grad, positives=RAMP's ℓ∞ & ℓ₁ adv, negs=other-class adv, τ=0.1, α=β=0.5)
- **B2 = RAMP + worst-case SupCon** (γ·SupCon on the ℓ∞ adv embeddings; γ small ~0.2)
Projection head discarded at eval. Seed 0. Backbone saved as a RAMP-family state_dict → audited on the PC with the frozen 12-AA harness.

## How to run
1. Upload **`attackdro_code.zip`** (repo root; contains `scripts/dev/claim_b_rep.py` + `patch_ramp_claimb.py`) to Drive `MyDrive/attackdro/`.
2. Runtime → GPU. Edit the `EDIT ME` paths. Run all. It clones RAMP (public), applies the patch, **smoke-tests 1 epoch**, then trains B1 then B2 (80ep).
3. Download each `val_best.pth` from Drive → send to the PC for the identical 12-AA audit + paired B−R bootstrap.

⚠ **The RAMP integration is UNTESTED on the PC** (RAMP's env deps don't import there). The 1-epoch smoke cell validates it here BEFORE the 80-epoch runs — do not skip it.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## EDIT ME


In [ ]:
DRIVE_CODE_ZIP = '/content/drive/MyDrive/attackdro/attackdro_code.zip'   # has claim_b_rep.py + patch
DRIVE_OUT      = '/content/drive/MyDrive/attackdro/ClaimB_out'           # RAMP save_dir (val_best + logs land here)
LBD = 5        # RAMP KL weight — MATCH the config that produced your R bar (armA_rampfull). scratch-recipe default = 5.
import os; os.makedirs(DRIVE_OUT, exist_ok=True)


In [ ]:
# Clone RAMP (public) + deps. RAMP needs robustbench; torch/torchvision preinstalled on Colab.
![ -d /content/RAMP ] || git clone https://github.com/uiuc-focal-lab/RAMP /content/RAMP
!pip -q install robustbench 2>/dev/null; pip -q install wandb 2>/dev/null
print('RAMP cloned + deps installed')


In [ ]:
# Unpack our code; put claim_b_rep.py where RAMP_claimB.py will import it; set CLAIMB_REP_DIR.
import zipfile, shutil, os
os.makedirs('/content/attackdro', exist_ok=True)
with zipfile.ZipFile(DRIVE_CODE_ZIP) as z: z.extractall('/content/attackdro')
shutil.copy('/content/attackdro/scripts/dev/claim_b_rep.py', '/content/RAMP/claim_b_rep.py')
shutil.copy('/content/attackdro/scripts/dev/patch_ramp_claimb.py', '/content/RAMP/patch_ramp_claimb.py')
os.environ['CLAIMB_REP_DIR'] = '/content/RAMP'
print('claim_b_rep + patch in /content/RAMP')


In [ ]:
# Apply the patch -> /content/RAMP/RAMP_claimB.py ; verify all 6 injections + syntax
%cd /content/RAMP
!python patch_ramp_claimb.py
import ast; ast.parse(open('/content/RAMP/RAMP_claimB.py').read()); print('RAMP_claimB.py SYNTAX OK')


## SMOKE — 1 epoch B1 (validate the injected loss + val_best on Colab BEFORE the long runs)


In [ ]:
AT,EP,SD = 2,1,'/content/smoke'
%cd /content/RAMP
import os
cmd = (f"CLAIMB_REP_DIR=/content/RAMP python RAMP_claimB.py --lr-max 0.05 --lr-schedule=static "
       f"--at_iter {AT} --epochs {EP} --save_freq 10 --eval_freq 10 --fname B1_smoke "
       f"--kl --max --gp --lbd {LBD} --seed 0 --claimB B1 --cb_alpha 0.5 --cb_beta 0.5 "
       f"--data_dir /content/data --save_dir '{SD}'")
print(cmd)
get_ipython().system(cmd)
print('SMOKE val_best written:', os.path.exists('/content/smoke/B1_smoke/val_best.pth'))


## Full B1 = RAMP + pull-push (80 ep, seed 0) → Drive


In [ ]:
AT,EP,SD = 10,80,DRIVE_OUT
%cd /content/RAMP
import os
cmd = (f"CLAIMB_REP_DIR=/content/RAMP python RAMP_claimB.py --lr-max 0.05 --lr-schedule=static "
       f"--at_iter {AT} --epochs {EP} --save_freq 10 --eval_freq 10 --fname B1_pullpush_seed0 "
       f"--kl --max --gp --lbd {LBD} --seed 0 --claimB B1 --cb_alpha 0.5 --cb_beta 0.5 "
       f"--data_dir /content/data --save_dir '{SD}'")
print(cmd)
get_ipython().system(cmd)


## Full B2 = RAMP + worst-case SupCon (80 ep, seed 0) → Drive


In [ ]:
AT,EP,SD = 10,80,DRIVE_OUT
%cd /content/RAMP
import os
cmd = (f"CLAIMB_REP_DIR=/content/RAMP python RAMP_claimB.py --lr-max 0.05 --lr-schedule=static "
       f"--at_iter {AT} --epochs {EP} --save_freq 10 --eval_freq 10 --fname B2_supcon_seed0 "
       f"--kl --max --gp --lbd {LBD} --seed 0 --claimB B2 --cb_gamma 0.2 "
       f"--data_dir /content/data --save_dir '{SD}'")
print(cmd)
get_ipython().system(cmd)


AT,EP,SD = 10,80,DRIVE_OUT
%cd /content/RAMP
import os
cmd = (f"CLAIMB_REP_DIR=/content/RAMP python RAMP_claimB.py --lr-max 0.05 --lr-schedule=static "
       f"--at_iter {AT} --epochs {EP} --save_freq 10 --eval_freq 10 --fname B1_pullpush_seed0 "
       f"--kl --max --gp --lbd {LBD} --seed 0 --claimB B1 --cb_alpha 0.5 --cb_beta 0.5 "
       f"--data_dir /content/data --save_dir '{SD}'")
print(cmd)
get_ipython().system(cmd)
